<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 El Cazador de Cumbres en la Niebla: el Mapa Mental y las Excursiones Caras 🏔️
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/07%20-%20Seleccion%20de%20Modelos%20y%20Optimizacion/Para%20Dummies/03_Optimizacion_Bayesiana_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 03 (Optimización bayesiana)](../03_Optimizacion_Bayesiana.ipynb). Buscar las mejores «perillas» de un modelo es como **buscar la cumbre más alta de una cordillera en medio de la niebla**: no ves el paisaje; solo puedes **ir a un lugar y medir su altura**, y **cada excursión es carísima** (entrenar un modelo puede tardar horas). ¿Cómo elegir a dónde ir **sabiendo lo que ya midiste**? Aquí **no hay fórmulas**: usamos una herramienta estadística de *scikit-learn* (un «proceso gaussiano») como **caja**: tú le das la lista de excursiones hechas y ella te devuelve **cuánto espera** y **cuánto duda** en cada punto de la cordillera. (Cómo funciona por dentro se explica en el cuaderno estándar.)

Al terminar podrás explicar:
1. Qué es el **mapa mental** (modelo sustituto) y por qué ayuda a **no desperdiciar excursiones**.
2. El dilema de **explorar o explotar** (¿voy a lo conocido bueno o a un lugar misterioso?) con el ejemplo del **restaurante**.
3. Cuándo la optimización bayesiana **vale la pena** (cuando cada excursión es cara) y cuándo **no** (cuando ir es barato).

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../03_Optimizacion_Bayesiana.ipynb). Antes: [Buscar las llaves (Dummies)](02_Random_Grid_Search_Dummies.ipynb).

---
## 1. El mapa mental: aprender de cada excursión 🧠🗺️

Reglas del juego: la cordillera mide **10 km**. Puedes hacer **pocas excursiones**; en cada una vas a un punto y mides la **altura**. Dos formas de decidir:

- **Al azar**: cada excursión a un lugar cualquiera, sin recordar lo anterior.
- **Con mapa mental**: después de cada excursión **actualizas tu idea del paisaje**: «entre dos puntos medidos, la altura **cambia suavemente**» (tu **estimación**), y «**dudo más cuanto más lejos esté de lo ya medido**» (tu **duda**). Luego vas al lugar con **más promesa**: **estimación alta + curiosidad × duda**.

Veamos el mapa mental trabajar, **paso a paso**. La línea punteada es la cordillera real (el cazador **no la ve**), la azul es su estimación, la banda es su duda y la estrella es **a dónde irá la siguiente excursión**:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel

x_mapa = np.linspace(0, 10, 501)
def cordillera(x, a=2.5, ha=0.75, wa=1.3, b=7.0, wb=0.8):
    """Una colina ancha y baja (a), una cumbre más estrecha y MAS alta (b) y un poco de ondulación."""
    return ha * np.exp(-((x - a) / wa) ** 2) + 1.0 * np.exp(-((x - b) / wb) ** 2) + 0.04 * np.sin(3 * x)

def mapa_mental(visitados_x, visitados_h, curiosidad):
    """Estimación y duda en cada punto (caja: proceso gaussiano de scikit-learn) y 'atractivo' = estimación + curiosidad * duda."""
    caja = GaussianProcessRegressor(ConstantKernel(1.0, "fixed") * RBF(1.0, "fixed"), alpha=1e-6, optimizer=None, normalize_y=True)
    caja.fit(visitados_x[:, None], visitados_h)
    estimacion, duda = caja.predict(x_mapa[:, None], return_std=True)
    return estimacion, duda, estimacion + curiosidad * duda

visitados_x = np.array([1.0, 4.5, 9.0]); visitados_h = cordillera(visitados_x)         # 3 excursiones iniciales (lejos de la cumbre más alta)
fig, axes = plt.subplots(2, 3, figsize=(16, 7.5), sharex=True, sharey=True)
for paso, ax in enumerate(axes.ravel(), start=1):
    est, duda, atractivo = mapa_mental(visitados_x, visitados_h, curiosidad=2.0)
    siguiente = x_mapa[np.argmax(atractivo)]
    ax.plot(x_mapa, cordillera(x_mapa), "k--", lw=1, label="Cordillera real (oculta)")
    ax.plot(x_mapa, est, color="#2563eb", lw=2, label="Estimación del cazador"); ax.fill_between(x_mapa, est - 2 * duda, est + 2 * duda, color="#2563eb", alpha=0.2, label="Duda")
    ax.scatter(visitados_x, visitados_h, color="black", zorder=5, label="Excursiones hechas")
    ax.scatter([siguiente], [cordillera(siguiente)], marker="*", s=260, color="#dc2626", zorder=6, label="Próxima excursión")
    ax.set_ylim(-0.6, 1.6); ax.set_title(f"Paso {paso}: mejor altura medida = {visitados_h.max():.2f}", fontweight="bold", fontsize=10)
    if paso == 1: ax.legend(fontsize=7, loc="upper left")
    visitados_x = np.append(visitados_x, siguiente); visitados_h = np.append(visitados_h, cordillera(siguiente))
for ax in axes[1]: ax.set_xlabel("Posición en la cordillera (km)")
for ax in axes[:, 0]: ax.set_ylabel("Altura")
plt.tight_layout(); plt.show()
cumbre = cordillera(x_mapa).max()
print(f"Cumbre real = {cumbre:.2f} | mejor altura tras 3 + 6 = 9 excursiones guiadas = {visitados_h[:9].max():.2f}")
assert visitados_h[:9].max() > 0.9 * cumbre            # con 3 + 6 excursiones guiadas ya se llegó a la cumbre

---
**Lo que se ve:** al principio el cazador explora cerca de lo que midió y de los **huecos dudosos** (la banda es ancha lejos de los puntos negros). Cuando el hueco entre la colina baja y el extremo derecho se vuelve el lugar más prometedor, **descubre la cumbre más alta** y se queda cerca de ella. Fíjate: las excursiones **no** se desperdician repitiendo lugares ya medidos (la duda ahí es cero).

## 2. ¿Gana al azar? Mil cordilleras distintas 🎲

Una sola cordillera no demuestra nada (¿y si tuvo suerte?). Generamos **300 cordilleras distintas** (con una colina baja y una cumbre más alta en lugares y tamaños aleatorios) y comparamos, con **10 excursiones** cada uno, al cazador **al azar** contra el cazador **con mapa mental** (3 excursiones iniciales al azar + 7 guiadas):

In [ ]:
rng = np.random.default_rng(1)
juegos, excursiones = 300, 10
cumbre_juego, curva_azar, curva_mapa = [], [], []
for _ in range(juegos):
    a, b = rng.uniform(1, 4), rng.uniform(5.5, 9)                                          # lugares de las dos colinas
    ha, wa, wb = rng.uniform(0.5, 0.9), rng.uniform(1.0, 1.6), rng.uniform(0.6, 1.0)       # alturas y anchos
    f = lambda x: cordillera(x, a, ha, wa, b, wb); cumbre_juego.append(f(x_mapa).max())
    # Al azar: 10 excursiones sin memoria
    curva_azar.append(np.maximum.accumulate(f(rng.uniform(0, 10, excursiones))))
    # Con mapa mental: 3 al azar y 7 guiadas (curiosidad 2)
    xm = rng.uniform(0, 10, 3); hm = f(xm)
    for _ in range(excursiones - 3):
        _, _, atractivo = mapa_mental(xm, hm, 2.0)
        nuevo = x_mapa[np.argmax(atractivo)]; xm = np.append(xm, nuevo); hm = np.append(hm, f(nuevo))
    curva_mapa.append(np.maximum.accumulate(hm))
cumbre_juego = np.array(cumbre_juego); curva_azar, curva_mapa = np.array(curva_azar), np.array(curva_mapa)
frac_azar, frac_mapa = curva_azar[:, -1] / cumbre_juego, curva_mapa[:, -1] / cumbre_juego
print(f"Tras 10 excursiones, fracción de la cumbre real alcanzada -> al azar: {frac_azar.mean():.0%} | con mapa mental: {frac_mapa.mean():.0%}")
print(f"El mapa mental llega a >= 95 % de la cumbre en el {np.mean(frac_mapa >= 0.95):.0%} de las cordilleras; el azar, en el {np.mean(frac_azar >= 0.95):.0%}.")

fig, ax = plt.subplots(figsize=(9, 4.4))
for curva, col, nom in [(curva_azar, "#dc2626", "Al azar"), (curva_mapa, "#16a34a", "Con mapa mental")]:
    r = (curva / cumbre_juego[:, None]).mean(axis=0); ax.plot(np.arange(1, excursiones + 1), r, "o-", color=col, lw=2, label=nom)
ax.axvline(3.5, color="gray", ls=":", label="Fin de las 3 excursiones iniciales (al azar)")
ax.set_xlabel("Excursiones hechas"); ax.set_ylabel("Fracción de la cumbre real alcanzada (promedio)"); ax.legend(fontsize=8)
ax.set_title("300 cordilleras: aprender de lo medido ahorra excursiones", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert frac_mapa.mean() > frac_azar.mean() + 0.03

---
**Lección:** con la **misma cantidad de excursiones**, el cazador con mapa mental se acerca más a la cumbre (y llega a ≥ 95 % de ella bastante más seguido). **Una advertencia honesta:** esta cordillera es una **cordillera de una sola dimensión con dos colinas**, hecha para que se note la ventaja. En problemas reales con **pocas perillas y mucho margen** (como las dos perillas de un SVM del cuaderno estándar), el azar ya llega **cerca de lo máximo** y la ventaja es **pequeña**.

## 3. Explorar o explotar: el dilema del restaurante 🍽️

Acabas de mudarte. Conoces **un restaurante bueno**. Cada noche decides: **volver al conocido** (seguro: *explotar*) o **probar uno nuevo** (podría ser mejor o peor: *explorar*). Si **siempre** repites el conocido, **nunca descubres** el mejor; si **siempre** pruebas uno nuevo, **nunca disfrutas** lo que ya sabes. La **curiosidad** del cazador es esa perilla: con **0**, solo va a donde cree que hay altura (¡se queda **atascado** en lo ya medido!); con **mucha**, solo recorre lugares nuevos (sin importar si parecen buenos). Probemos varios niveles de curiosidad:

In [ ]:
rng = np.random.default_rng(2)
curiosidades = [0.0, 0.5, 2.0, 4.0, 10.0]
juegos_c = 100
resultados = {c: [] for c in curiosidades}; azar_c = []
for _ in range(juegos_c):
    a, b = rng.uniform(1, 4), rng.uniform(5.5, 9); ha, wa, wb = rng.uniform(0.5, 0.9), rng.uniform(1.0, 1.6), rng.uniform(0.6, 1.0)
    f = lambda x: cordillera(x, a, ha, wa, b, wb); cumbre_j = f(x_mapa).max()
    x0 = rng.uniform(0, 10, 3)                                            # las mismas 3 excursiones iniciales para todas las curiosidades
    azar_c.append(f(rng.uniform(0, 10, 10)).max() / cumbre_j)
    for c in curiosidades:
        xm, hm = x0.copy(), f(x0)
        for _ in range(7):
            _, _, atractivo = mapa_mental(xm, hm, c); nuevo = x_mapa[np.argmax(atractivo)]
            xm = np.append(xm, nuevo); hm = np.append(hm, f(nuevo))
        resultados[c].append(hm.max() / cumbre_j)
medias = {c: np.mean(v) for c, v in resultados.items()}
for c, m in medias.items(): print(f"Curiosidad {c:>4}: alcanza el {m:.0%} de la cumbre en promedio")
print(f"(Referencia: 10 excursiones al azar alcanzan el {np.mean(azar_c):.0%})")
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(range(len(curiosidades)), list(medias.values()), "o-", color="#7c3aed", lw=2, label="Con mapa mental")
ax.axhline(np.mean(azar_c), color="#dc2626", ls="--", label="Al azar (10 excursiones)")
ax.set_xticks(range(len(curiosidades))); ax.set_xticklabels([str(c) for c in curiosidades]); ax.legend(fontsize=8)
ax.set_xlabel("Curiosidad (0 = solo explotar; muy alta = solo explorar)"); ax.set_ylabel("Fracción de la cumbre alcanzada (10 excursiones)")
ax.set_title("Ni solo explotar ni solo explorar: hay un punto medio", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
mejor_c = max(medias, key=medias.get)
assert medias[0.0] < medias[mejor_c] and 0.0 < mejor_c < 10.0
print(f"La mejor curiosidad en esta simulación fue {mejor_c}: ni 0 (se queda cerca de lo que ya midió) ni muy alta (solo pasea). En el cuaderno estándar esa perilla se llama κ (kappa) o ξ (xi).")

---
**Lección:** con **curiosidad 0** el cazador **apenas mejora al azar**: se queda cerca de lo que cree mejor y casi nunca revisa los huecos. Con **curiosidad enorme** recorre todo el mapa sin mirar lo que ya sabe. **El punto medio gana.** Es el mismo dilema de los bandidos del módulo de refuerzo, pero aquí el «restaurante» es un punto de una escala continua.

## 4. ¿Cuándo vale la pena? Cuando cada excursión es cara 💸

El mapa mental cuesta **un poco de cálculo** (pensar). Vale la pena **solo si cada excursión es mucho más cara que pensar**. Contamos cuántas excursiones necesita cada método para llegar a **95 % de la cumbre** (en las 300 cordilleras de antes) y lo convertimos en dinero con distintos precios por excursión:

In [ ]:
objetivo = 0.95
def excursiones_hasta(curvas, cumbres):
    """Primera excursion en la que el mejor valor llega al objetivo (si no llega en 10, se cuenta 11)."""
    llega = (curvas / cumbres[:, None]) >= objetivo
    return np.where(llega.any(axis=1), llega.argmax(axis=1) + 1, excursiones + 1)

n_azar, n_mapa = excursiones_hasta(curva_azar, cumbre_juego), excursiones_hasta(curva_mapa, cumbre_juego)
print(f"Excursiones (mediana) para llegar al {objetivo:.0%} de la cumbre -> al azar: {np.median(n_azar):.0f} | con mapa mental: {np.median(n_mapa):.0f}")
print(f"Media -> al azar: {n_azar.mean():.1f} | con mapa mental: {n_mapa.mean():.1f}  (11 = no llegó en 10 excursiones)")
ahorro = n_azar.mean() - n_mapa.mean()
print(f"\n{'Costo de cada excursión':>26} | {'ahorro promedio por búsqueda':>30}")
for costo in (0.01, 1, 100, 10000):
    print(f"{costo:>24,.2f}$ | {ahorro * costo:>28,.2f}$")
print("\nSi una excursión cuesta centavos (entrenar un modelo diminuto), el «pensar» no se paga: prueba al azar. Si cuesta horas de computador, ahorrar excursiones es oro.")
assert ahorro > 0

---
**Lección:** la optimización bayesiana **ahorra excursiones**; eso es valioso **si cada excursión es cara** (entrenar una red grande durante horas) y casi irrelevante si es barata (un modelo pequeño que entrena en un segundo: ahí basta el azar). También **funciona mejor con pocas perillas** (como un mapa de 10 km, no uno de mil dimensiones) y **sufre si las mediciones son ruidosas** (la altura que mides cambia cada vez).

## Resumen en una frase 🎯

> **Cuando cada prueba de un modelo es cara, conviene aprender de cada excursión: se construye un mapa mental de lo que se espera y de dónde se duda, y se va al lugar más prometedor, balanceando curiosidad (explorar) y aprovechar lo conocido (explotar); con pruebas baratas o pocas perillas, el azar ya basta.**

### Lo que aprendiste hoy:
- ✅ **Mapa mental** = estimación + duda; se actualiza con **cada** excursión y evita repetir lo ya medido.
- ✅ **300 cordilleras**: con las mismas 10 excursiones, el cazador con mapa mental se acercó más a la cumbre que el azar .
- ✅ **Explorar o explotar**: curiosidad 0 apenas mejora al azar; curiosidad enorme solo pasea; **el punto medio gana**.
- ✅ **¿Vale la pena?** Solo cuando cada excursión **cuesta mucho**; con pruebas baratas, probar al azar es suficiente.

> 🎓 **Cierre del módulo (y del Capítulo 2):** ahora puedes explorar el [cuaderno estándar](../03_Optimizacion_Bayesiana.ipynb), donde el mapa mental es un **proceso gaussiano real** y la «curiosidad» se llama **mejora esperada**, y continuar con el [Taller Práctico](../../homeworks/07_Seleccion_Modelos_Optimizacion_Hands_On.ipynb).

---
##### 🛠️ Práctica 1: Tu propia cordillera: ¿a dónde irá el cazador?

Pon al cazador a trabajar en una cordillera **distinta** a las del cuaderno. Usa `cordillera()`, `mapa_mental()` y `x_mapa` (ya definidos). La cordillera nueva es `cordillera(x, 3.0, 0.7, 1.3, 6.5, 0.8)` (colina baja en 3.0 y cumbre más alta en 6.5):

1. Empieza con tres excursiones: `visitados_x = np.array([1.0, 5.0, 9.0])` y mide sus alturas `visitados_h`.
2. Repite **8 veces**: pide el `atractivo` con `mapa_mental(visitados_x, visitados_h, 2.0)` (la curiosidad es 2), ve al punto de `x_mapa` donde es máximo, mide su altura y añádelos a tus listas.
3. Calcula `mejor_final` (la mayor altura medida) y `cumbre` (la altura máxima de la cordillera sobre `x_mapa`).
4. Comprueba con `assert` que el cazador **no empeoró** (`mejor_final` ≥ la mejor altura de las 3 excursiones iniciales) y que llegó al menos al **90 %** de la cumbre real.

In [ ]:
# Escribe tu código aquí

# f_nueva = lambda x: cordillera(x, 3.0, 0.7, 1.3, 6.5, 0.8)
# visitados_x = np.array([1.0, 5.0, 9.0]); visitados_h = f_nueva(visitados_x)
# mejor_inicial = visitados_h.max()
# for _ in range(8):
#     _, _, atractivo = mapa_mental(visitados_x, visitados_h, 2.0)
#     siguiente = x_mapa[np.argmax(...)]
#     visitados_x = np.append(visitados_x, siguiente)
#     visitados_h = np.append(visitados_h, f_nueva(...))
# mejor_final = visitados_h.max()
# cumbre = f_nueva(x_mapa).max()


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
f_nueva = lambda x: cordillera(x, 3.0, 0.7, 1.3, 6.5, 0.8)      # colina baja en 3.0, cumbre mas alta en 6.5

visitados_x = np.array([1.0, 5.0, 9.0]); visitados_h = f_nueva(visitados_x)
mejor_inicial = visitados_h.max()

for _ in range(8):
    _, _, atractivo = mapa_mental(visitados_x, visitados_h, 2.0)     # estimacion + 2 * duda en cada punto
    siguiente = x_mapa[np.argmax(atractivo)]                         # ir donde el atractivo es mayor
    visitados_x = np.append(visitados_x, siguiente)
    visitados_h = np.append(visitados_h, f_nueva(siguiente))         # medir la altura (la parte "cara")

mejor_final = visitados_h.max()
cumbre = f_nueva(x_mapa).max()
print("Excursiones guiadas (km):", np.round(visitados_x[3:], 2))
print(f"Mejor altura inicial = {mejor_inicial:.2f} | mejor final = {mejor_final:.2f} | cumbre real = {cumbre:.2f} ({mejor_final / cumbre:.0%})")

assert mejor_final >= mejor_inicial
assert mejor_final >= 0.9 * cumbre
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué diferencia hay entre buscar **al azar** y buscar **con un mapa mental**? ¿Qué parte de lo medido se «aprovecha» en la segunda?
2. Piensa en elegir **dónde cenar** durante un mes en una ciudad nueva. ¿Cómo repartirías las noches entre **probar sitios nuevos** y **repetir los buenos**? ¿Cambiaría si solo vas a estar una semana?
3. Tu modelo entrena en **2 segundos**. ¿Usarías optimización bayesiana o probarías mil combinaciones al azar? ¿Y si entrena **3 horas** y solo puedes hacer 20 pruebas?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>